# DriveRender Setup


In [ ]:
import os
import sys
import subprocess
import time
import json
import urllib.request
from pathlib import Path
from google.colab import drive

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/V3ct0R924/DriveRender/main"
DRIVE_RENDER = "/content/drive/MyDrive/DriveRender"
CONFIG_DIR = f"{DRIVE_RENDER}/config"
LANGS_DIR = f"{CONFIG_DIR}/langs"

try:
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    try:
        with urllib.request.urlopen(f"{GITHUB_RAW_BASE}/langs/en.json") as resp:
            boot_lang = json.load(resp)
        print(boot_lang['setup']['drive_mount_error'].format(error=e))
    except Exception:
        print(f"[setup.drive_mount_error] {e}")
    raise

os.makedirs(LANGS_DIR, exist_ok=True)

urllib.request.urlretrieve(
    f"{GITHUB_RAW_BASE}/drive_render_common.py",
    f"{CONFIG_DIR}/drive_render_common.py",
)
urllib.request.urlretrieve(
    f"{GITHUB_RAW_BASE}/langs/en.json",
    f"{LANGS_DIR}/en.json",
)

if CONFIG_DIR not in sys.path:
    sys.path.insert(0, CONFIG_DIR)

from drive_render_common import (
    Color,
    DEFAULT_LANGUAGE,
    SUPPORTED_LANGUAGES,
    BLENDER_DIR,
    OUTPUT_DIR,
    PROJECTS_DIR,
    TEMP_DIR,
    CONFIG_PATH,
    ensure_dirs,
    get_blender_version,
    load_language_dict,
    make_translator,
)

LANG = load_language_dict(DEFAULT_LANGUAGE)
T = make_translator(LANG)
print(T('setup', 'download_assets'))

urllib.request.urlretrieve(
    f"{GITHUB_RAW_BASE}/langs/es.json",
    f"{LANGS_DIR}/es.json",
)

dirs = ensure_dirs()
FOLDER_LABELS = {
    'config': 'folder_config',
    'blender': 'folder_blender',
    'projects': 'folder_projects',
    'output': 'folder_output',
    'temp': 'folder_temp',
}

USER_CONFIG = {}
IS_FIRST_RUN = False
CONFIG_FILE = CONFIG_PATH

print(f"\n{Color.BOLD}{Color.HEADER}")
print("██████╗ ██████╗ ██╗██╗   ██╗███████╗    ██████╗ ███████╗███╗   ██╗██████╗ ███████╗██████╗ ")
print("██╔══██╗██╔══██╗██║██║   ██║██╔════╝    ██╔══██╗██╔════╝████╗  ██║██╔══██╗██╔════╝██╔══██╗")
print("██║  ██║██████╔╝██║██║   ██║█████╗      ██████╔╝█████╗  ██╔██╗ ██║██║  ██║█████╗  ██████╔╝")
print("██║  ██║██╔══██╗██║╚██╗ ██╔╝██╔══╝      ██╔══██╗██╔══╝  ██║╚██╗██║██║  ██║██╔══╝  ██╔══██╗")
print("██████╔╝██║  ██║██║ ╚████╔╝ ███████╗    ██║  ██║███████╗██║ ╚████║██████╔╝███████╗██║  ██║")
print("╚═════╝ ╚═╝  ╚═╝╚═╝  ╚═══╝  ╚══════╝    ╚═╝  ╚═╝╚══════╝╚═╝  ╚═══╝╚═════╝ ╚══════╝╚═╝  ╚═╝")
print(f"{Color.ENDC}")
print(f"{Color.OKCYAN}{T('setup', 'subtitle')}{Color.ENDC}")
print("=" * 90)
print()

print(f"{Color.BOLD}{T('setup', 'section_drive')}{Color.ENDC}")
print("-" * 90)
print(f"{Color.OKGREEN}{T('setup', 'drive_mounted')}{Color.ENDC}")

print(f"{Color.OKCYAN}{T('setup', 'waiting')}{Color.ENDC}")
time.sleep(5)
print()

print(f"{Color.BOLD}{T('setup', 'section_hardware')}{Color.ENDC}")
print("-" * 90)
gpu_available = False
try:
    gpu_info = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        encoding='utf-8',
    ).strip()
    gpu_name, gpu_vram = gpu_info.split(',')
    print(f"{Color.OKGREEN}{T('setup', 'gpu_detected', name=gpu_name.strip())}{Color.ENDC}")
    print(f"{Color.OKGREEN}{T('setup', 'gpu_vram', vram=gpu_vram.strip())}{Color.ENDC}")
    gpu_available = True
except Exception:
    print(f"{Color.FAIL}{T('setup', 'no_gpu')}{Color.ENDC}")
    print(f"{Color.WARNING}{T('setup', 'slow_without_gpu')}{Color.ENDC}")
    print(f"{Color.OKBLUE}{T('setup', 'gpu_solution')}{Color.ENDC}")

ram_info = subprocess.check_output(['free', '-h'], encoding='utf-8')
ram_total = ram_info.split('\n')[1].split()[1]
print(f"{Color.OKGREEN}{T('setup', 'ram_total', total=ram_total)}{Color.ENDC}")

print(f"{Color.OKCYAN}{T('setup', 'waiting')}{Color.ENDC}")
time.sleep(5)
print()

print(f"{Color.BOLD}{T('setup', 'section_dirs')}{Color.ENDC}")
print("-" * 90)
print(T('setup', 'creating_dirs'))
for key, path in dirs.items():
    if key == 'base':
        continue
    label = T('setup', FOLDER_LABELS.get(key, key))
    if os.path.exists(path):
        print(f"{Color.OKBLUE}{T('setup', 'folder_exists', name=label, path=path)}{Color.ENDC}")
    else:
        print(f"{Color.OKGREEN}{T('setup', 'folder_created', name=label, path=path)}{Color.ENDC}")

print(f"{Color.BOLD}{T('setup', 'section_config')}{Color.ENDC}")

if not os.path.exists(CONFIG_FILE):
    IS_FIRST_RUN = True
    print(f"{Color.WARNING}{T('setup', 'first_run_no_config')}{Color.ENDC}")
    print(T('setup', 'first_run_choose_lang'))
    lang = input(f"{Color.BOLD}{T('setup', 'lang_prompt')}{Color.ENDC}").strip().lower()
    if not lang:
        lang = DEFAULT_LANGUAGE
    if lang not in SUPPORTED_LANGUAGES:
        print(f"{Color.WARNING}{T('setup', 'lang_not_recognized')}{Color.ENDC}")
        lang = DEFAULT_LANGUAGE

    USER_CONFIG = {
        'language': lang,
        'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'last_updated': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    try:
        with open(CONFIG_FILE, 'w', encoding='utf-8') as f:
            json.dump(USER_CONFIG, f, ensure_ascii=False, indent=2)
        print(f"{Color.OKGREEN}{T('setup', 'config_created', path=CONFIG_FILE)}{Color.ENDC}")
    except Exception as e:
        print(f"{Color.FAIL}{T('setup', 'config_save_error', error=e)}{Color.ENDC}")
else:
    try:
        with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
            USER_CONFIG = json.load(f)
        print(f"{Color.OKGREEN}{T('setup', 'config_loaded', path=CONFIG_FILE)}{Color.ENDC}")
        if 'language' in USER_CONFIG:
            print(f"{Color.OKCYAN}{T('setup', 'language_set', language=USER_CONFIG['language'])}{Color.ENDC}")
    except Exception as e:
        print(f"{Color.FAIL}{T('setup', 'config_read_error', error=e)}{Color.ENDC}")
        USER_CONFIG = {}

USER_LANGUAGE = USER_CONFIG.get('language', DEFAULT_LANGUAGE)
if USER_LANGUAGE not in SUPPORTED_LANGUAGES:
    print(f"{Color.WARNING}{T('setup', 'invalid_language', language=USER_LANGUAGE)}{Color.ENDC}")
    USER_LANGUAGE = DEFAULT_LANGUAGE

LANG = load_language_dict(USER_LANGUAGE)
T = make_translator(LANG)

if IS_FIRST_RUN:
    print(f"{Color.OKBLUE}{T('setup', 'first_run_hint', path=CONFIG_FILE)}{Color.ENDC}")

print(f"\n{Color.OKGREEN}{T('setup', 'dirs_configured')}{Color.ENDC}")
print(f"{Color.OKCYAN}{T('setup', 'waiting')}{Color.ENDC}")
time.sleep(5)
print()

print(f"{Color.BOLD}{T('setup', 'section_detect')}{Color.ENDC}")
print("-" * 90)

BLENDER_VERSIONS = {
    '1': {
        'name': 'Blender 4.5 LTS',
        'url': 'https://download.blender.org/release/Blender4.5/blender-4.5.0-linux-x64.tar.xz',
        'folder': 'blender-4.5.0-linux-x64',
    },
    '2': {
        'name': 'Blender 5.0',
        'url': 'https://download.blender.org/release/Blender5.0/blender-5.0.0-linux-x64.tar.xz',
        'folder': 'blender-5.0.0-linux-x64',
    },
}

blender_installed = False
current_version = None
blender_executable = None

print(T('setup', 'scanning_blender'))
for key, ver in BLENDER_VERSIONS.items():
    blender_path = f"{BLENDER_DIR}/{ver['folder']}/blender"
    if os.path.exists(blender_path):
        version_output = get_blender_version(blender_path)
        if version_output is None:
            print(f"{Color.WARNING}{T('setup', 'install_not_executable', name=ver['name'])}{Color.ENDC}")
            print(f"{Color.WARNING}{T('setup', 'path_label', path=blender_path)}{Color.ENDC}")
            continue
        blender_installed = True
        current_version = ver['name']
        blender_executable = blender_path
        print(f"{Color.OKGREEN}{T('setup', 'install_found', name=ver['name'])}{Color.ENDC}")
        print(f"{Color.OKBLUE}{T('setup', 'location_label', path=blender_path)}{Color.ENDC}")
        print(f"{Color.OKBLUE}{T('setup', 'version_label', version=version_output)}{Color.ENDC}")
        break

if not blender_installed:
    print(f"{Color.WARNING}{T('setup', 'no_previous_install')}{Color.ENDC}")

print(f"{Color.OKCYAN}{T('setup', 'waiting')}{Color.ENDC}")
time.sleep(5)
print()

print(f"{Color.BOLD}{T('setup', 'section_install')}{Color.ENDC}")
print("-" * 90)

if not blender_installed:
    print(f"{Color.WARNING}{T('setup', 'install_required')}{Color.ENDC}\n")
    print(T('setup', 'versions_available'))
    for key, ver in BLENDER_VERSIONS.items():
        print(f"  [{key}] {ver['name']}")

    selection = input(f"\n{Color.BOLD}{T('setup', 'select_version')}{Color.ENDC}").strip()

    if selection in BLENDER_VERSIONS:
        selected = BLENDER_VERSIONS[selection]
        print(f"\n{Color.OKBLUE}{T('setup', 'download_start', name=selected['name'])}{Color.ENDC}")
        print(f"{Color.WARNING}{T('setup', 'download_may_take')}{Color.ENDC}\n")

        tar_file = f"{TEMP_DIR}/blender_temp.tar.xz"
        try:
            subprocess.run(['wget', '-O', tar_file, selected['url'], '--show-progress'], check=True)
            print(f"\n{Color.OKGREEN}{T('setup', 'download_complete')}{Color.ENDC}")

            print(f"{Color.OKBLUE}{T('setup', 'extracting')}{Color.ENDC}")
            subprocess.run(['tar', '-xf', tar_file, '-C', BLENDER_DIR], check=True)
            print(f"{Color.OKGREEN}{T('setup', 'extract_complete')}{Color.ENDC}")

            os.remove(tar_file)
            print(f"{Color.OKBLUE}{T('setup', 'temp_removed')}{Color.ENDC}")

            blender_executable = f"{BLENDER_DIR}/{selected['folder']}/blender"
            if os.path.exists(blender_executable):
                version_output = get_blender_version(blender_executable)
                if version_output is None:
                    print(f"{Color.FAIL}{T('setup', 'install_no_version')}{Color.ENDC}")
                else:
                    print(f"\n{Color.OKGREEN}{T('setup', 'install_success')}{Color.ENDC}")
                    print(f"{Color.OKGREEN}{T('setup', 'install_ready', name=selected['name'])}{Color.ENDC}")
                    print(f"{Color.OKBLUE}{T('setup', 'version_label', version=version_output)}{Color.ENDC}")
                    current_version = selected['name']
            else:
                print(f"{Color.FAIL}{T('setup', 'install_failed')}{Color.ENDC}")
        except Exception as e:
            print(f"{Color.FAIL}{T('setup', 'install_error', error=e)}{Color.ENDC}")
    else:
        print(f"{Color.FAIL}{T('setup', 'invalid_selection')}{Color.ENDC}")
else:
    print(f"{Color.OKGREEN}{T('setup', 'already_configured', version=current_version)}{Color.ENDC}")
    print(f"{Color.OKBLUE}{T('setup', 'executable_label', path=blender_executable)}{Color.ENDC}")

print()
print('=' * 90)
print(f"{Color.BOLD}{Color.OKGREEN}{T('setup', 'setup_complete')}{Color.ENDC}")
print('=' * 90)

gpu_status_text = T('setup', 'gpu_available') if gpu_available else T('setup', 'gpu_unavailable')
gpu_color = Color.OKGREEN if gpu_available else Color.FAIL
if current_version:
    blender_color = Color.OKGREEN
    blender_status_text = current_version
else:
    blender_color = Color.WARNING
    blender_status_text = T('setup', 'blender_not_installed')

print(f"\n{Color.BOLD}{T('setup', 'system_summary')}{Color.ENDC}")
print(f"  GPU: {gpu_color}{gpu_status_text}{Color.ENDC}")
print(f"  Blender: {blender_color}{blender_status_text}{Color.ENDC}")
print(f"  {T('setup', 'render_folder', path=OUTPUT_DIR)}")
print(f"  {T('setup', 'projects_folder', path=PROJECTS_DIR)}")
print(f"\n{Color.BOLD}{Color.OKCYAN}{T('setup', 'ready_to_render')}{Color.ENDC}")
print()


# Select File


In [ ]:
import os
import sys
import time
from pathlib import Path

CONFIG_DIR = "/content/drive/MyDrive/DriveRender/config"
if CONFIG_DIR not in sys.path:
    sys.path.insert(0, CONFIG_DIR)

from drive_render_common import (
    Color,
    PROJECTS_DIR,
    SELECTED_FILE_PATH,
    ensure_dirs,
    get_user_language,
    load_language_dict,
    make_translator,
    write_selected_blend,
)

USER_LANGUAGE = globals().get('USER_LANGUAGE') or get_user_language()
LANG = load_language_dict(USER_LANGUAGE)
T = make_translator(LANG)
dirs = globals().get('dirs') or ensure_dirs()

def print_section(title, width=90):
    print(f"\n{Color.BOLD}{title}{Color.ENDC}")
    print("-" * width)

def print_success(msg):
    print(f"{Color.OKGREEN}{msg}{Color.ENDC}")

def print_error(msg):
    print(f"{Color.FAIL}{msg}{Color.ENDC}")

def print_warning(msg):
    print(f"{Color.WARNING}{msg}{Color.ENDC}")

def print_info(msg):
    print(f"{Color.OKBLUE}{msg}{Color.ENDC}")

print_section(T('select', 'section_title'))

projects_dir = Path(dirs['projects'])

print_info(T('select', 'scanning'))
time.sleep(1)

files = sorted([
    f for f in projects_dir.iterdir()
    if f.is_file()
    and not f.name.startswith('.')
    and not f.name.startswith('~')
])

blend_files = [f for f in files if f.suffix.lower() == '.blend']
other_files = [f for f in files if f.suffix.lower() != '.blend']

if not files:
    print_error(T('select', 'no_files', path=projects_dir))
    print_warning(T('select', 'upload_hint'))
    print_info(T('select', 'full_path', path=projects_dir))
    SELECTED_FILE = None
else:
    print(f"\n{Color.BOLD}{T('select', 'files_found', count=len(files))}{Color.ENDC}\n")

    file_list = blend_files + other_files

    for i, f in enumerate(file_list, start=1):
        size_mb = f.stat().st_size / (1024 * 1024)
        if f.suffix.lower() == '.blend':
            print(f"{Color.OKGREEN}[{i:2d}]{Color.ENDC} {Color.BOLD}{f.name}{Color.ENDC} "
                  f"{Color.OKCYAN}({size_mb:.2f} MB){Color.ENDC}")
        else:
            print(f"{Color.WARNING}[{i:2d}]{Color.ENDC} {f.name} "
                  f"{Color.OKCYAN}({size_mb:.2f} MB){Color.ENDC} "
                  f"{Color.WARNING}{T('select', 'not_blend_tag')}{Color.ENDC}")

    print(f"\n{'-' * 90}")

    while True:
        try:
            choice = input(
                f"{Color.BOLD}{T('select', 'enter_number', max=len(file_list))}{Color.ENDC}"
            ).strip()

            if not choice:
                print_warning(T('select', 'must_enter_number'))
                continue

            idx = int(choice)

            if 1 <= idx <= len(file_list):
                selected_file = file_list[idx - 1]

                if selected_file.suffix.lower() != '.blend':
                    print_warning(T('select', 'not_blend_warning', name=selected_file.name))
                    confirm = input(
                        f"{Color.WARNING}{T('select', 'continue_anyway')}{Color.ENDC}"
                    ).strip().lower()
                    if confirm not in ['s', 'si', 'yes', 'y']:
                        print_info(T('select', 'selection_cancelled'))
                        continue

                break
            else:
                print_error(T('select', 'out_of_range', max=len(file_list)))

        except ValueError:
            print_error(T('select', 'invalid_input'))
        except KeyboardInterrupt:
            print_error(T('select', 'cancelled_by_user'))
            SELECTED_FILE = None
            raise

    SELECTED_FILE = str(selected_file)

    print(f"\n{Color.OKGREEN}{'=' * 90}{Color.ENDC}")
    print_success(T('select', 'file_selected', name=selected_file.name))
    print_info(T('select', 'full_path', path=SELECTED_FILE))
    print_success(T('select', 'file_size', size=f"{selected_file.stat().st_size / (1024 * 1024):.2f}"))
    print(f"{Color.OKGREEN}{'=' * 90}{Color.ENDC}")

    try:
        write_selected_blend(
            SELECTED_FILE,
            metadata={
                'filename': selected_file.name,
                'size_mb': round(selected_file.stat().st_size / (1024 * 1024), 2),
                'selected_at': time.strftime('%Y-%m-%d %H:%M:%S'),
            },
        )
        print_info(T('select', 'selection_saved', path=SELECTED_FILE_PATH))
    except Exception as e:
        print_warning(T('select', 'save_failed'))
        print_warning(str(e))
        print_info(T('select', 'session_available'))

print(f"\n{Color.OKCYAN}{T('select', 'variable_ready')}{Color.ENDC}\n")


# Analyze Selected File


In [ ]:
import shutil
import json
import subprocess
import sys
from pathlib import Path
import time

CONFIG_DIR = "/content/drive/MyDrive/DriveRender/config"
if CONFIG_DIR not in sys.path:
    sys.path.insert(0, CONFIG_DIR)

from drive_render_common import (
    Color,
    BLENDER_DIR,
    TEMP_DIR,
    clean_temp_dir,
    ensure_dirs,
    get_blender_version,
    get_user_language,
    load_language_dict,
    make_translator,
    read_selected_blend_path,
)

USER_LANGUAGE = globals().get('USER_LANGUAGE') or get_user_language()
LANG = load_language_dict(USER_LANGUAGE)
T = make_translator(LANG)
dirs = globals().get('dirs') or ensure_dirs()

def print_section(title, width=90):
    print(f"\n{Color.BOLD}{title}{Color.ENDC}")
    print("-" * width)

def print_success(msg):
    print(f"{Color.OKGREEN}✓ {msg}{Color.ENDC}")

def print_error(msg):
    print(f"{Color.FAIL}✗ {msg}{Color.ENDC}")

def print_warning(msg):
    print(f"{Color.WARNING}⚠ {msg}{Color.ENDC}")

def print_info(msg):
    print(f"{Color.OKBLUE}ℹ {msg}{Color.ENDC}")

def print_data(label, value, indent=2):
    spaces = " " * indent
    print(f"{spaces}{Color.OKCYAN}{label}:{Color.ENDC} {Color.BOLD}{value}{Color.ENDC}")

def find_blender(blender_dir):
    blender_path = Path(blender_dir)
    if not blender_path.exists():
        return None, None
    for item in blender_path.iterdir():
        if item.is_dir() and 'blender' in item.name.lower():
            blender_exe = item / 'blender'
            if blender_exe.exists() and blender_exe.is_file():
                return str(blender_exe), str(item)
    return None, None

print_section(T('analyze', 'section_title'))

selected_path = globals().get('SELECTED_FILE') or read_selected_blend_path()
if not selected_path:
    print_error(T('analyze', 'no_file_selected'))
    print_info(T('analyze', 'run_select_first'))
    raise ValueError("SELECTED_FILE is not defined")

SELECTED_FILE = selected_path
blend_path = Path(SELECTED_FILE)

print_info(T('analyze', 'validating', name=blend_path.name))
time.sleep(1)

if not blend_path.exists():
    print_error(T('analyze', 'file_not_found', path=blend_path))
    raise FileNotFoundError(T('analyze', 'file_does_not_exist', path=blend_path))

if blend_path.suffix.lower() != '.blend':
    print_warning(T('analyze', 'not_blend_ext', ext=blend_path.suffix))

print_success(T('analyze', 'valid_file', name=blend_path.name))
print_data(T('analyze', 'size_label'), f"{blend_path.stat().st_size / (1024 * 1024):.2f} MB")

print("\n" + "-" * 90)
print_info(T('analyze', 'locating_blender'))
time.sleep(1)

blender_exe = shutil.which('blender')

if not blender_exe:
    print_warning(T('analyze', 'blender_not_in_path'))
    print_info(T('analyze', 'searching_blender_dir'))
    blender_exe, blender_dir = find_blender(dirs['blender'])
    if blender_exe:
        print_success(T('analyze', 'blender_found', path=blender_dir))
    else:
        print_error(T('analyze', 'blender_not_in_dir'))
        print_info(T('analyze', 'searched_dir', path=dirs['blender']))

if not blender_exe:
    print_error(T('analyze', 'blender_not_located'))
    print_info(T('analyze', 'run_setup_first'))
    raise FileNotFoundError("Blender executable not found")

print_success(T('analyze', 'blender_at', path=blender_exe))

version_output = get_blender_version(blender_exe)
if version_output:
    print_data(T('analyze', 'version_label'), version_output)
else:
    print_warning(T('analyze', 'could_not_verify_version'))

print("\n" + "-" * 90)
print_info(T('analyze', 'preparing_analysis'))
time.sleep(1)

blender_script = """
import bpy
import json

bpy.ops.wm.open_mainfile(filepath=BLEND_PATH_PLACEHOLDER)

info = {}
scene = bpy.context.scene
engine = scene.render.engine
info['engine'] = engine
info['engine_changed'] = False

if engine != 'CYCLES':
    info['engine_original'] = engine
    scene.render.engine = 'CYCLES'
    info['engine_forced'] = 'CYCLES'
    info['engine_changed'] = True
    info['engine_warning'] = "Engine '" + engine + "' changed to CYCLES"

if scene.render.engine == 'CYCLES':
    cycles = scene.cycles
    info['cycles_config'] = {
        'device': cycles.device,
        'samples': cycles.samples if hasattr(cycles, 'samples') else cycles.preview_samples,
        'adaptive_sampling': cycles.use_adaptive_sampling,
        'denoising': cycles.use_denoising if hasattr(cycles, 'use_denoising') else 'unknown',
        'max_bounces': cycles.max_bounces,
        'diffuse_bounces': cycles.diffuse_bounces,
        'glossy_bounces': cycles.glossy_bounces,
        'transparent_bounces': cycles.transparent_max_bounces,
    }

info['resolution'] = {
    'width': scene.render.resolution_x,
    'height': scene.render.resolution_y,
    'percentage': scene.render.resolution_percentage,
    'fps': scene.render.fps,
    'frame_start': scene.frame_start,
    'frame_end': scene.frame_end,
    'frame_current': scene.frame_current,
}

total_verts = total_edges = total_faces = total_tris = 0
total_quads = total_ngons = total_objects = mesh_objects = 0

for ob in bpy.data.objects:
    total_objects += 1
    if ob.type == 'MESH' and ob.data is not None:
        mesh_objects += 1
        me = ob.data
        total_verts += len(me.vertices)
        total_edges += len(me.edges)
        total_faces += len(me.polygons)
        me.calc_loop_triangles()
        total_tris += len(me.loop_triangles)
        for poly in me.polygons:
            vert_count = len(poly.vertices)
            if vert_count == 4:
                total_quads += 1
            elif vert_count > 4:
                total_ngons += 1

info['geometry'] = {
    'total_objects': total_objects,
    'mesh_objects': mesh_objects,
    'vertices': total_verts,
    'edges': total_edges,
    'faces': total_faces,
    'triangles': total_tris,
    'quads': total_quads,
    'ngons': total_ngons,
}

info['scene_info'] = {
    'name': scene.name,
    'camera': scene.camera.name if scene.camera else None,
    'world': scene.world.name if scene.world else None,
}

info['materials'] = {
    'count': len(bpy.data.materials),
    'count_with_nodes': sum(1 for m in bpy.data.materials if m.use_nodes),
}

info['textures'] = {
    'images': len(bpy.data.images),
    'textures': len(bpy.data.textures),
}

if info['engine_changed']:
    try:
        bpy.ops.wm.save_mainfile()
        info['file_saved'] = True
    except Exception as e:
        info['file_saved'] = False
        info['save_error'] = str(e)

print('BLEND_INFO_START')
print(json.dumps(info))
print('BLEND_INFO_END')
"""

blender_script = blender_script.replace('BLEND_PATH_PLACEHOLDER', json.dumps(str(blend_path)))

print("\n" + "-" * 90)
print_info(T('analyze', 'running_analysis'))

script_path = None
try:
    script_path = Path(TEMP_DIR) / f"analyze_{int(time.time())}.py"
    script_path.write_text(blender_script, encoding='utf-8')
    print_info(T('analyze', 'temp_script', path=script_path))

    process_start = time.time()
    output = subprocess.check_output(
        [blender_exe, '--background', '--python', str(script_path)],
        stderr=subprocess.STDOUT,
        encoding='utf-8',
        timeout=300,
    )
    process_time = time.time() - process_start
    print_success(T('analyze', 'analysis_complete', seconds=process_time))

    print("\n" + "-" * 90)
    print_info(T('analyze', 'processing_results'))
    time.sleep(1)

    json_data = None
    capture = False
    json_lines = []
    for line in output.split('\n'):
        if 'BLEND_INFO_START' in line:
            capture = True
            continue
        if 'BLEND_INFO_END' in line:
            capture = False
            break
        if capture:
            json_lines.append(line)

    if json_lines:
        try:
            json_data = json.loads(''.join(json_lines))
        except json.JSONDecodeError as e:
            print_error(T('analyze', 'json_parse_error', error=e))
            print_warning(T('analyze', 'blender_full_output'))
            print(output)

    if not json_data:
        print_error(T('analyze', 'extract_failed'))
        print_warning(T('analyze', 'blender_output'))
        print(output)
        raise ValueError("Analysis failed")

    print("\n" + "=" * 90)
    print(f"{Color.BOLD}{Color.HEADER}{T('analyze', 'results_title')}{Color.ENDC}")
    print("=" * 90)

    print(f"\n{Color.BOLD}{T('analyze', 'render_engine_section')}{Color.ENDC}")
    if json_data.get('engine_changed'):
        print_warning(T('analyze', 'original_engine', engine=json_data.get('engine_original')))
        print_success(T('analyze', 'forced_engine', engine=json_data.get('engine_forced')))
        if json_data.get('engine_warning'):
            print_warning(json_data.get('engine_warning'))
        if json_data.get('file_saved'):
            print_success(T('analyze', 'changes_saved'))
        else:
            print_error(T('analyze', 'save_failed', error=json_data.get('save_error', '?')))
    else:
        print_success(T('analyze', 'current_engine', engine=json_data.get('engine')))

    if 'cycles_config' in json_data:
        print(f"\n{Color.BOLD}{T('analyze', 'cycles_section')}{Color.ENDC}")
        cycles = json_data['cycles_config']
        on_off = lambda v: T('analyze', 'enabled') if v else T('analyze', 'disabled')
        print_data(T('analyze', 'device_label'), cycles['device'])
        print_data(T('analyze', 'samples_label'), cycles['samples'])
        print_data(T('analyze', 'adaptive_sampling_label'), on_off(cycles['adaptive_sampling']))
        print_data(T('analyze', 'denoising_label'), on_off(cycles['denoising']))
        print_data(T('analyze', 'max_bounces_label'), cycles['max_bounces'])
        print_data(T('analyze', 'diffuse_bounces_label'), cycles['diffuse_bounces'])
        print_data(T('analyze', 'glossy_bounces_label'), cycles['glossy_bounces'])
        print_data(T('analyze', 'transparent_bounces_label'), cycles['transparent_bounces'])

    if 'resolution' in json_data:
        print(f"\n{Color.BOLD}{T('analyze', 'resolution_section')}{Color.ENDC}")
        res = json_data['resolution']
        print_data(T('analyze', 'resolution_label'), f"{res['width']} x {res['height']} ({res['percentage']}%)")
        print_data(T('analyze', 'fps_label'), res['fps'])
        total_frames = res['frame_end'] - res['frame_start'] + 1
        print_data(
            T('analyze', 'frames_label'),
            T('analyze', 'frames_value', start=res['frame_start'], end=res['frame_end'], total=total_frames),
        )

    if 'geometry' in json_data:
        print(f"\n{Color.BOLD}{T('analyze', 'geometry_section')}{Color.ENDC}")
        geo = json_data['geometry']
        print_data(T('analyze', 'total_objects'), f"{geo['total_objects']:,}")
        print_data(T('analyze', 'mesh_objects'), f"{geo['mesh_objects']:,}")
        print_data(T('analyze', 'vertices'), f"{geo['vertices']:,}")
        print_data(T('analyze', 'edges'), f"{geo['edges']:,}")
        print_data(T('analyze', 'faces'), f"{geo['faces']:,}")
        print_data(T('analyze', 'triangles'), f"{geo['triangles']:,}")
        print_data(T('analyze', 'quads'), f"{geo['quads']:,}")
        print_data(T('analyze', 'ngons'), f"{geo['ngons']:,}")

    if 'scene_info' in json_data:
        print(f"\n{Color.BOLD}{T('analyze', 'scene_section')}{Color.ENDC}")
        scene = json_data['scene_info']
        print_data(T('analyze', 'name_label'), scene['name'])
        print_data(T('analyze', 'camera_label'), scene['camera'] or T('analyze', 'not_set'))
        print_data(T('analyze', 'world_label'), scene['world'] or T('analyze', 'not_set'))

    if 'materials' in json_data and 'textures' in json_data:
        print(f"\n{Color.BOLD}{T('analyze', 'materials_section')}{Color.ENDC}")
        print_data(
            T('analyze', 'materials_count'),
            T('analyze', 'materials_value',
              count=json_data['materials']['count'],
              nodes=json_data['materials']['count_with_nodes']),
        )
        print_data(T('analyze', 'images_label'), f"{json_data['textures']['images']:,}")
        print_data(T('analyze', 'textures_label'), f"{json_data['textures']['textures']:,}")

    print("\n" + "=" * 90)
    print_success(T('analyze', 'analysis_success'))
    print(f"{Color.OKCYAN}{T('analyze', 'ready_to_render')}{Color.ENDC}")
    print("=" * 90 + "\n")

except subprocess.CalledProcessError as e:
    print_error(T('analyze', 'blender_error_code', code=e.returncode))
    print_warning(T('analyze', 'process_output'))
    print(e.output)
    raise

except subprocess.TimeoutExpired:
    print_error(T('analyze', 'analysis_timeout'))
    print_info(T('analyze', 'file_too_complex'))
    raise

except Exception as e:
    print_error(T('analyze', 'unexpected_error', type=type(e).__name__))
    print_error(str(e))
    raise

finally:
    if script_path and Path(script_path).exists():
        try:
            Path(script_path).unlink()
        except Exception as e:
            print_warning(T('analyze', 'temp_clean_failed', error=e))
    clean_temp_dir()
    print_info(T('analyze', 'temp_cleaned'))


# Render


In [ ]:
import time
import subprocess
import shutil
import json
import sys
from pathlib import Path
from datetime import datetime

CONFIG_DIR = "/content/drive/MyDrive/DriveRender/config"
if CONFIG_DIR not in sys.path:
    sys.path.insert(0, CONFIG_DIR)

from drive_render_common import (
    Color,
    TEMP_DIR,
    clean_temp_dir,
    detect_gpu_compute_backends,
    ensure_dirs,
    get_user_language,
    load_language_dict,
    make_translator,
    read_selected_blend_path,
    sanitize_filename,
)

USER_LANGUAGE = globals().get('USER_LANGUAGE') or get_user_language()
LANG = load_language_dict(USER_LANGUAGE)
T = make_translator(LANG)
dirs = globals().get('dirs') or ensure_dirs()

RENDER_TIMEOUT = 6 * 3600

def print_section(title, width=90):
    print(f"\n{Color.BOLD}{title}{Color.ENDC}")
    print("-" * width)

def print_success(msg):
    print(f"{Color.OKGREEN}✓ {msg}{Color.ENDC}")

def print_error(msg):
    print(f"{Color.FAIL}✗ {msg}{Color.ENDC}")

def print_warning(msg):
    print(f"{Color.WARNING}⚠ {msg}{Color.ENDC}")

def print_info(msg):
    print(f"{Color.OKBLUE}ℹ {msg}{Color.ENDC}")

def print_progress(msg):
    print(f"{Color.OKCYAN}▶ {msg}{Color.ENDC}")

def format_time(seconds):
    if seconds < 60:
        return T('render', 'format_seconds', seconds=seconds)
    if seconds < 3600:
        return T('render', 'format_minutes', mins=int(seconds // 60), secs=int(seconds % 60))
    return T('render', 'format_hours', hours=int(seconds // 3600), mins=int((seconds % 3600) // 60))

def find_blender(blender_dir):
    blender_path = Path(blender_dir)
    if not blender_path.exists():
        return None, None
    for item in blender_path.iterdir():
        if item.is_dir() and 'blender' in item.name.lower():
            blender_exe = item / 'blender'
            if blender_exe.exists() and blender_exe.is_file():
                return str(blender_exe), str(item)
    return None, None

print(f"\n{Color.BOLD}{Color.HEADER}")
print("██████╗ ███████╗███╗   ██╗██████╗ ███████╗██████╗     ███████╗████████╗ █████╗ ██████╗ ████████╗")
print("██╔══██╗██╔════╝████╗  ██║██╔══██╗██╔════╝██╔══██╗    ██╔════╝╚══██╔══╝██╔══██╗██╔══██╗╚══██╔══╝")
print("██████╔╝█████╗  ██╔██╗ ██║██║  ██║█████╗  ██████╔╝    ███████╗   ██║   ███████║██████╔╝   ██║   ")
print("██╔══██╗██╔══╝  ██║╚██╗██║██║  ██║██╔══╝  ██╔══██╗    ╚════██║   ██║   ██╔══██║██╔══██╗   ██║   ")
print("██║  ██║███████╗██║ ╚████║██████╔╝███████╗██║  ██║    ███████║   ██║   ██║  ██║██║  ██║   ██║   ")
print("╚═╝  ╚═╝╚══════╝╚═╝  ╚═══╝╚═════╝ ╚══════╝╚═╝  ╚═╝    ╚══════╝   ╚═╝   ╚═╝  ╚═╝╚═╝  ╚═╝   ╚═╝   ")
print(f"{Color.ENDC}")
print(f"{Color.OKCYAN}{T('render', 'subtitle')}{Color.ENDC}")
print("=" * 90)

print_section(T('render', 'section_validation'))

selected_path = globals().get('SELECTED_FILE') or read_selected_blend_path()
if not selected_path:
    print_error(T('render', 'no_blend_selected'))
    print_info(T('render', 'run_select_cell'))
    raise ValueError("SELECTED_FILE is not defined")

SELECTED_FILE = selected_path
blend_path = Path(SELECTED_FILE)

if not blend_path.exists():
    print_error(T('render', 'file_not_exist', path=blend_path))
    raise FileNotFoundError(T('render', 'file_not_found', path=blend_path))

print_success(T('render', 'file_found', name=blend_path.name))
print_info(T('render', 'file_size', size=f"{blend_path.stat().st_size / (1024 * 1024):.2f}"))

output_dir = Path(dirs['output'])
output_dir.mkdir(parents=True, exist_ok=True)
print_success(T('render', 'output_dir', path=output_dir))

blender_exe = shutil.which('blender')
if not blender_exe:
    print_warning(T('render', 'blender_not_in_path'))
    print_info(T('render', 'searching_versions'))
    blender_exe, _ = find_blender(dirs['blender'])
    if not blender_exe:
        print_error(T('render', 'blender_exe_not_found'))
        raise FileNotFoundError("Blender executable not found")

print_success(T('render', 'blender_found', path=blender_exe))

gpu_backends = detect_gpu_compute_backends()
print_info(T('render', 'gpu_backends', backends=', '.join(gpu_backends)))

print("\n" + "=" * 90)
print(f"{Color.BOLD}{T('render', 'render_summary')}{Color.ENDC}")
print(f"  {T('render', 'summary_file')}: {Color.BOLD}{blend_path.name}{Color.ENDC}")
print(f"  {T('render', 'summary_size')}: {blend_path.stat().st_size / (1024 * 1024):.2f} MB")
print(f"  {T('render', 'summary_engine')}: {Color.BOLD}{T('render', 'engine_cycles')}{Color.ENDC}")
print(f"  {T('render', 'summary_output')}: {output_dir}")
print("=" * 90)

print(f"\n{Color.WARNING}{T('render', 'render_warning')}{Color.ENDC}")
print(T('render', 'warn_complexity'))
print(T('render', 'warn_samples'))
print(T('render', 'warn_resolution'))
print(T('render', 'warn_gpu'))
print()

while True:
    resp = input(f"{Color.BOLD}{T('render', 'confirm_render')}{Color.ENDC}").strip().lower()
    if resp in ['s', 'si', 'yes', 'y']:
        print_success(T('render', 'render_confirmed'))
        break
    if resp in ['n', 'no']:
        print_warning(T('render', 'render_cancelled'))
        raise KeyboardInterrupt("Render cancelled")
    print_error(T('render', 'invalid_confirm'))

print_section(T('render', 'section_preparation'))

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
blend_name = sanitize_filename(blend_path.stem)
out_file = output_dir / f"{blend_name}_cycles_{timestamp}.png"

print_info(T('render', 'output_file', name=out_file.name))
print_progress(T('render', 'generating_script'))

quality_presets = {
    "draft": {
        "label_key": "preset_draft",
        "samples": 64,
        "resolution_percentage": 50,
        "max_bounces": 4,
        "diffuse_bounces": 2,
        "glossy_bounces": 2,
        "transparent_max_bounces": 4,
        "use_adaptive_sampling": True,
        "use_denoising": True,
        "use_simplify": True,
        "simplify_subdivision_render": 1,
        "simplify_child_particles": 0.5,
    },
    "final": {
        "label_key": "preset_final",
        "samples": 256,
        "resolution_percentage": 100,
        "max_bounces": 8,
        "diffuse_bounces": 4,
        "glossy_bounces": 4,
        "transparent_max_bounces": 8,
        "use_adaptive_sampling": True,
        "use_denoising": True,
        "use_simplify": False,
        "simplify_subdivision_render": 2,
        "simplify_child_particles": 1.0,
    },
}

print(f"\n{T('render', 'quality_options')}")
print(T('render', 'quality_draft_desc'))
print(T('render', 'quality_final_desc'))

while True:
    quality = input(f"{Color.BOLD}{T('render', 'choose_quality')}{Color.ENDC}").strip().lower()
    if quality == "":
        quality = "draft"
    if quality in quality_presets:
        break
    print_error(T('render', 'invalid_quality'))

preset = quality_presets[quality].copy()
preset['name'] = T('render', preset['label_key'])
del preset['label_key']
preset['compute_device_types'] = gpu_backends
print_success(T('render', 'quality_selected', name=preset['name']))

preset_json = json.dumps(preset)

blender_script = """
import bpy
import json
import sys
from pathlib import Path

preset = json.loads(PRESET_JSON_PLACEHOLDER)

def print_progress(message):
    print(f"RENDER_PROGRESS: {message}", flush=True)

def configure_gpu(scene):
    compute_types = preset.get('compute_device_types') or ['OPTIX', 'CUDA']
    try:
        prefs = bpy.context.preferences
        cycles_prefs = prefs.addons['cycles'].preferences
        for compute_type in compute_types:
            try:
                cycles_prefs.compute_device_type = compute_type
                if hasattr(cycles_prefs, 'get_devices'):
                    cycles_prefs.get_devices()
                enabled = False
                for device in cycles_prefs.devices:
                    if device.type == 'CPU':
                        continue
                    device.use = True
                    enabled = True
                if enabled:
                    scene.cycles.device = 'GPU'
                    print_progress('GPU backend: ' + compute_type)
                    return compute_type
            except Exception:
                continue
    except Exception as e:
        print_progress('GPU setup failed: ' + str(e))
    scene.cycles.device = 'CPU'
    print_progress('Using CPU')
    return None

try:
    print_progress('Opening .blend file...')
    bpy.ops.wm.open_mainfile(filepath=BLEND_PATH_PLACEHOLDER)
    print_progress('File opened')

    scene = bpy.context.scene
    print_progress('Scene: ' + scene.name)

    original_engine = scene.render.engine
    print_progress('Original engine: ' + str(original_engine))

    if original_engine != 'CYCLES':
        scene.render.engine = 'CYCLES'
        print_progress("Engine changed to CYCLES")

    configure_gpu(scene)
    cycles = scene.cycles

    samples = preset.get('samples')
    if samples is not None and hasattr(cycles, 'samples'):
        cycles.samples = samples

    if 'use_adaptive_sampling' in preset and hasattr(cycles, 'use_adaptive_sampling'):
        cycles.use_adaptive_sampling = bool(preset['use_adaptive_sampling'])

    if 'use_denoising' in preset and hasattr(cycles, 'use_denoising'):
        cycles.use_denoising = bool(preset['use_denoising'])

    if 'max_bounces' in preset:
        cycles.max_bounces = preset['max_bounces']
    if 'diffuse_bounces' in preset:
        cycles.diffuse_bounces = preset['diffuse_bounces']
    if 'glossy_bounces' in preset:
        cycles.glossy_bounces = preset['glossy_bounces']
    if 'transparent_max_bounces' in preset and hasattr(cycles, 'transparent_max_bounces'):
        cycles.transparent_max_bounces = preset['transparent_max_bounces']

    if 'use_simplify' in preset and hasattr(scene.render, 'use_simplify'):
        scene.render.use_simplify = bool(preset['use_simplify'])
        if hasattr(scene.render, 'simplify_subdivision_render') and 'simplify_subdivision_render' in preset:
            scene.render.simplify_subdivision_render = preset['simplify_subdivision_render']
        if hasattr(scene.render, 'simplify_child_particles') and 'simplify_child_particles' in preset:
            scene.render.simplify_child_particles = preset['simplify_child_particles']

    if 'resolution_percentage' in preset:
        scene.render.resolution_percentage = preset['resolution_percentage']

    print_progress('Preset: ' + str(preset.get('name', 'unknown')))
    if hasattr(cycles, 'samples'):
        print_progress('Samples: ' + str(cycles.samples))
    print_progress('Resolution: ' + str(scene.render.resolution_x) + 'x' + str(scene.render.resolution_y))

    outfile = OUTFILE_PLACEHOLDER
    Path(outfile).parent.mkdir(parents=True, exist_ok=True)

    scene.render.filepath = outfile
    scene.render.image_settings.file_format = 'PNG'
    scene.render.image_settings.color_mode = 'RGBA'

    print_progress('Output: ' + str(outfile))
    print_progress('Rendering current frame...')
    bpy.ops.render.render(write_still=True)
    print_progress('Render complete')

    if Path(outfile).exists():
        file_size = Path(outfile).stat().st_size / (1024 * 1024)
        print_progress('Saved: ' + format(file_size, '.2f') + ' MB')
        result = {
            'success': True,
            'outfile': outfile,
            'size_mb': file_size,
            'resolution': [scene.render.resolution_x, scene.render.resolution_y],
            'engine': scene.render.engine,
        }
        print('RENDER_RESULT_JSON')
        print(json.dumps(result))
        print('RENDER_RESULT_JSON_END')
    else:
        print_progress('ERROR: output file missing')
        sys.exit(1)

except Exception as e:
    print_progress('ERROR: ' + str(e))
    import traceback
    traceback.print_exc()
    sys.exit(1)
"""

blender_script = (
    blender_script.replace("PRESET_JSON_PLACEHOLDER", json.dumps(preset_json))
    .replace("BLEND_PATH_PLACEHOLDER", json.dumps(str(blend_path)))
    .replace("OUTFILE_PLACEHOLDER", json.dumps(str(out_file)))
)

script_path = None
process = None
try:
    script_path = Path(TEMP_DIR) / f"render_{int(time.time())}.py"
    script_path.write_text(blender_script, encoding='utf-8')
    print_success(T('render', 'script_generated'))

    print_section(T('render', 'section_execution'))
    print_warning(T('render', 'may_take_time'))
    print_info(T('render', 'press_ctrl_c'))
    print()

    render_start = time.time()
    deadline = render_start + RENDER_TIMEOUT

    process = subprocess.Popen(
        [blender_exe, '--background', '--python', str(script_path)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        encoding='utf-8',
        bufsize=1,
    )

    json_capture = False
    json_lines = []

    print(f"{Color.OKCYAN}{'─' * 90}{Color.ENDC}")

    while True:
        if time.time() > deadline:
            process.kill()
            process.wait()
            raise subprocess.TimeoutExpired(
                cmd=[blender_exe, '--background', '--python', str(script_path)],
                timeout=RENDER_TIMEOUT,
            )

        line = process.stdout.readline()
        if not line:
            if process.poll() is not None:
                break
            continue

        line = line.rstrip()
        if not line:
            continue

        if 'RENDER_RESULT_JSON' in line and 'END' not in line:
            json_capture = True
            continue
        if 'RENDER_RESULT_JSON_END' in line:
            json_capture = False
            continue
        if json_capture:
            json_lines.append(line)
            continue

        if 'RENDER_PROGRESS:' in line:
            print_progress(line.split('RENDER_PROGRESS:', 1)[1].strip())
        elif 'Fra:' in line:
            print(f"{Color.OKCYAN}{line}{Color.ENDC}")
        elif 'Remaining:' in line or 'Sample' in line:
            print(f"{Color.OKBLUE}{line}{Color.ENDC}")
        elif 'Error' in line or 'ERROR' in line:
            print_error(line)
        elif 'Warning' in line or 'WARNING' in line:
            print_warning(line)

    return_code = process.wait()
    print(f"{Color.OKCYAN}{'─' * 90}{Color.ENDC}\n")
    render_time = time.time() - render_start

    print_section(T('render', 'section_verification'))

    if return_code != 0:
        print_error(T('render', 'blender_exit_code', code=return_code))
        raise subprocess.CalledProcessError(return_code, blender_exe)

    result_data = None
    if json_lines:
        try:
            result_data = json.loads(''.join(json_lines))
        except json.JSONDecodeError as e:
            print_warning(T('render', 'json_parse_warning', error=e))

    if out_file.exists():
        file_size = out_file.stat().st_size / (1024 * 1024)

        print("\n" + "=" * 90)
        print(f"{Color.BOLD}{Color.OKGREEN}{T('render', 'render_success')}{Color.ENDC}")
        print("=" * 90)

        print(f"\n{Color.BOLD}{T('render', 'render_details')}{Color.ENDC}")
        print(f"  {T('render', 'detail_file')}: {Color.BOLD}{out_file.name}{Color.ENDC}")
        print(f"  {T('render', 'detail_size')}: {Color.OKGREEN}{file_size:.2f} MB{Color.ENDC}")
        print(f"  {T('render', 'detail_location')}: {Color.OKBLUE}{out_file}{Color.ENDC}")
        print(f"  {T('render', 'detail_time')}: {Color.OKCYAN}{format_time(render_time)}{Color.ENDC}")

        if result_data:
            if 'resolution' in result_data:
                res = result_data['resolution']
                print(f"  {T('render', 'detail_resolution')}: {res[0]} x {res[1]}")
            if 'engine' in result_data:
                print(f"  {T('render', 'detail_engine')}: {result_data['engine']}")

        print("\n" + "=" * 90)
        print(f"{Color.OKGREEN}{T('render', 'saved_in', dir=output_dir)}{Color.ENDC}")
        print("=" * 90 + "\n")

        try:
            from IPython.display import Image, display
            print(f"{Color.BOLD}{T('render', 'preview_title')}{Color.ENDC}\n")
            display(Image(filename=str(out_file)))
        except Exception:
            print_info(T('render', 'download_from_drive'))
    else:
        print_error(T('render', 'output_not_generated'))
        print_warning(T('render', 'check_errors'))
        raise FileNotFoundError(T('render', 'output_not_found', path=out_file))

except subprocess.CalledProcessError as e:
    print_error(T('render', 'blender_errors', code=e.returncode))
    print_warning(T('render', 'check_error_messages'))
    raise

except subprocess.TimeoutExpired:
    print_error(T('render', 'render_timeout', hours=RENDER_TIMEOUT // 3600))
    print_info(T('render', 'process_terminated'))
    raise

except KeyboardInterrupt:
    print_warning(T('render', 'cancelled'))
    print_info(T('render', 'interrupted'))
    if process is not None and process.poll() is None:
        process.terminate()
        process.wait()
    raise

except Exception as e:
    print_error(T('render', 'unexpected_error', type=type(e).__name__))
    print_error(str(e))
    raise

finally:
    if script_path and Path(script_path).exists():
        try:
            Path(script_path).unlink()
        except Exception:
            pass
    clean_temp_dir()
    print_info(T('render', 'temp_removed'))
